In [51]:
import pandas as pd
import os
import sys
import csv
import glob
import re
from datetime import datetime

# INPUTS (to be modified according to contract, quarter and language)
contract = '0044'
quarter = "2026 Q2"
chinese = "1"                   # "1" means data is in Chinese

# prior_to_change = "1"           # "1" means that if is prior to the further breakdown of the ad-supported stuff. So any report prior to 2024 should have "1"

outputdirectory = '../../50 KM Group/Royalties/Statements/Karen/_output/'

base_dir = f'../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_{contract}/{quarter}/as supplied by TME/'
outputfilename = f"../../50 KM Group/Royalties/Statements/Karen/_output/TME_{contract}_{quarter}.xlsx"
lookup_platform = '../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/support/lookup_platform.csv'
lookup_mv_album = '../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/support/lookup_album_mv.csv'  
lookup_fx = '../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/support/lookup_fx.csv' 

year = int(quarter.split()[0])


os.makedirs(outputdirectory, exist_ok=True)
logfile_name = (
    f'TME_{contract}_{quarter}_run_log_'
    + datetime.now().strftime('%Y%m%d_%H%M%S')
    + '.txt'
)
log_path = os.path.join(outputdirectory, logfile_name)
_log_file = open(log_path, 'w', encoding='utf-8')
_original_stdout = sys.stdout.streams[0] if type(sys.stdout).__name__ == '_Tee' else sys.stdout


class _Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for s in self.streams:
            s.write(data)
        self.flush()

    def flush(self):
        for s in self.streams:
            s.flush()

    def isatty(self):
        return False

    def __getattr__(self, name):
        return getattr(self.streams[0], name)


def ensure_logging():
    """Re-attach the run log. Jupyter replaces sys.stdout at the start of each cell."""
    global _original_stdout
    if _log_file is None or _log_file.closed:
        return
    stdout = sys.stdout
    if type(stdout).__name__ == '_Tee':
        if _log_file in getattr(stdout, 'streams', ()):
            return
        stdout = stdout.streams[0]
    _original_stdout = stdout
    sys.stdout = _Tee(_original_stdout, _log_file)


ensure_logging()


def close_log():
    sys.stdout.flush()
    if type(sys.stdout).__name__ == '_Tee':
        sys.stdout = sys.stdout.streams[0]
    else:
        sys.stdout = _original_stdout
    if _log_file and not _log_file.closed:
        _log_file.close()
    print(f'Run log saved to: {log_path}')


def fmt_int(n):
    try:
        return f'{int(n):,}'
    except (TypeError, ValueError):
        return str(n)


def fmt_money(n):
    try:
        return f'{float(n):,.4f}'.rstrip('0').rstrip('.')
    except (TypeError, ValueError):
        return str(n)


def header(title):
    line = '=' * 72
    print(f'\n{line}\n  {title}\n{line}')


def subheader(title):
    print(f'\n--- {title} ---')


header('TME quarterly processing')
print(f'  Run started              : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
print(f'  Contract                 : {contract}')
print(f'  Quarter                  : {quarter}')
print(f'  Chinese headers          : {chinese}')
print(f'  Output file              : {os.path.basename(outputfilename)}')
print(f'  Run log                  : {logfile_name}')


def find_date_range(filename):
    # Regular expression to match the date range format "YYYYMMDD-YYYYMMDD"
    pattern = r'\d{8}-\d{8}'
    # Search for the pattern in the filename
    match = re.search(pattern, filename)
    if match:
        return match.group()
    else:
        return None

subdirs_pattern = os.path.join(base_dir, '202*')
subdirs = list(set(glob.glob(subdirs_pattern)))

def read_and_combine_files(base_dir: str, keyword_file: str):
    base_dir.sort()
    df_all = []

    for subdir_per_month in base_dir:
        print(f'Accessing folder: {subdir_per_month}')
        df_list = []

        for files_in_one_month in os.listdir(subdir_per_month):
            if files_in_one_month.startswith(keyword_file):
                file_string = subdir_per_month + "/" + files_in_one_month
                df = pd.read_excel(file_string, sheet_name=0)
                print(file_string)
                if chinese == "1":
                    df['结算期间'] = find_date_range(file_string)
                else:
                    df['Period'] = find_date_range(file_string)
                df_list.append(df)
        if df_list:
            result_per_month = pd.concat(df_list, ignore_index=True)
            df_all.append(result_per_month)
    if df_all:
        final_df = pd.concat(df_all, ignore_index=True)
        return final_df
    else:
        return pd.DataFrame()  # Return an empty DataFrame if no relevant files were found

def merge(df1, df2, col):
    print(f"\nMerging on '{col}':")
    empty_cells = df1[col].isna().sum()
    print(f"There are a total of {empty_cells} rows that have no entry in {col}")
    df1.loc[:, col] = df1[col].fillna('n/a')
    #df1.fillna({col: 'n/a'}, inplace=True)
    df_merged = pd.merge(df1, df2, on=col, how='left')
    new_columns = df_merged.columns.difference(df1.columns)
    first_new_col = new_columns[0] if not new_columns.empty else None
    empty_cells2 = df_merged[first_new_col].isna().sum() if first_new_col else 0
    diff_empty = empty_cells2 - empty_cells
    if diff_empty == 0:
        print(f"Merging of column {col} was successful")
    else:
        print(f"Merging with issues. There are a total of {diff_empty} cells that could not be matched (see 'match_issues_{df1}_{col}.xlsx').")
        empty_rows = df_merged[df_merged[first_new_col].isna()]
        col=col.replace('/','')
        empty_rows.to_excel(f"{outputdirectory}/match_issues_{col}.xlsx", engine='openpyxl', index=False)
    print(f"The new dataframe has {df_merged.shape[0]} rows and {len(df_merged.columns)} columns.")
    return df_merged 

def add_columns_and_modify_columns_to_single_df(df_input:pd.DataFrame):
    df = df_input.copy()
    df.rename(columns={'License Fees - Total': 'Fee'}, inplace=True)
    df['Units'] = df['Sales_IOS'] + df['Sales_Others']
    df['UPC'] = pd.to_numeric(df['UPC'], errors='coerce')
    df['Composing Right End Date'] = df['Composing Right End Date'].astype(str)
    df['Composing Right Start Date'] = df['Composing Right Start Date'].astype(str)
    df['Product']='Purchase'
    return df

def add_columns_and_modify_columns_to_song_df(df_input:pd.DataFrame):
    df = df_input.copy()
    df['License Fees - Subscription Music Service(Senior)'] = pd.to_numeric(df['License Fees - Subscription Music Service(Senior)'], errors='coerce')
    df['License Fees - Subscription Music Service(Senior)'] = df['License Fees - Subscription Music Service(Senior)'].astype(float)
    return df

def add_columns_and_modify_columns_to_mv_df(df_input:pd.DataFrame):
    df = df_input.copy()
    df.rename(columns={'License Fees - Total': 'Fee'}, inplace=True)
    df.rename(columns={'Start Time of Authorization': 'Recording Right Start Date'}, inplace=True)
    df.rename(columns={'End Time of Authorization': 'Recording Right End Date'}, inplace=True)
    df.rename(columns={'Share of Recording Right': 'Right Share of Recorder'}, inplace=True)
    df.rename(columns={'MV': 'Song'}, inplace=True)
    df.rename(columns={'License Fees - Free Music Service-Non-free Mode-Consumption of MV': 'Units'}, inplace=True)
    if 'License Fees for the Ad-Supported Service—Non-free Mode' in df.columns:
        df = df.drop('License Fees for the Ad-Supported Service—Non-free Mode', axis=1)
    df['Product']='MV'
    return df

def add_columns_and_modify_columns_to_digalbum_df(df_input:pd.DataFrame):
    df = df_input.copy()
    df.rename(columns={'License Fees - Total': 'Fee'}, inplace=True)
    df['Units'] = df['Sales_IOS'] + df['Sales_Others']
    df = df.drop('Chargeable Digital Album Service',axis=1)
    df = df.drop('原始版权公司', axis=1)
    df = df.drop('版权公司名',axis=1)
    df = df.drop('版权公司歌曲编码',axis=1)
    df = df.drop('邻接权授权有效期开始',axis=1)
    df = df.drop('邻接权授权有效期结束',axis=1)
    df['Product']='Digital Album'
    return df

def add_columns_and_modify_columns_to_k_df(df_input:pd.DataFrame):
    df = df_input.copy()
    df.rename(columns={'License Fees - Total': 'Fee'}, inplace=True)
    #df.rename(columns={'License Fees-Per Play-Consumption of Recording-Original Version': 'Units'}, inplace=True)
    df['Units'] = df['License Fees-Per Play-Consumption of Lyrics'] + df['License Fees-Per Play-Consumption of Composition'] + df['License Fees-Per Play-Consumption of Recording-Original Version']
    df = df.drop('License Fees-Per Play-Consumption of Recording-Original Version', axis=1)
    df = df.drop('License Fees-Per Play-Consumption of Lyrics', axis=1)
    df = df.drop('License Fees-Per Play-Consumption of Composition', axis=1)
    df = df.drop('License Fees-Per Play-Consumption of Recording-Karaoke Version Provided by Licensor', axis=1)
    df = df.drop('License Fees-Per Play-Consumption of Recording-Karaoke Version Processed by TME', axis=1)
    df = df.drop('License Fees-Per Play', axis=1)
    df['Product']='Karaoke'
    return df

def rename_single_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
 #   '结算期间' : 'Period',
 #   '平台' : 'Platform',
 #   '日期' : 'Date',
 #   '中央曲库歌曲ID' : 'TMEID',
 #   '专辑名' : 'Album',
 #   '歌曲名' : 'Song',
 #   '歌手名' : 'Artist',
 #   '词作者' : 'Lyricist',
 #   '曲作者' : 'Composer',
 #   '专辑UPC' : 'UPC',
 #   '歌曲ISRC' : 'ISRC',
 #   '版权公司专辑编码' : 'Licensor Album-Code',
 #   '版权公司歌曲编码' : 'Licensor Code',
 #   '版权公司名' : 'Licensor',
 #   '原始版权公司' : 'Label',
 #   '词授权有效期开始' : 'Lyrics Right Start Date',
 #   '词授权有效期结束' : 'Lyrics Right End Date',
 #   '曲授权有效期开始' : 'Composing Right Start Date',
 #   '曲授权有效期结束' : 'Composing Right End Date',
 #   '邻接权授权有效期开始' : 'Recording Right Start Date',
 #   '邻接权授权有效期结束' : 'Recording Right End Date',
 #   '词份额' : 'Right Share of Writer',
 #   '曲份额' : 'Right Share of Composer',
 #   '邻接权份额' : 'Right Share of Recorder',
 #   '歌曲付费状态' : 'Charge Type',
 #   '单价' : 'Retail Price',
 #   'IOS销量' : 'Sales_IOS',
 #   '非IOS销量' : 'Sales_Others',
 #   '单曲订购金额' : "Single Track's Perchase - Revenue",
 #   '单曲订购收入分成' : 'License Fees - Single Track Purchase',
 #   'CP分成收入' : 'License Fees - Total'
     '结算期间' : 'Period',
     '平台' : 'Platform',
    '日期' : 'Date',
    '中央曲库歌曲ID' : 'TMEID',
    '专辑名' : 'Album',
    '歌曲名' : 'Song',
    '歌手名' : 'Artist',
    '词作者' : 'Lyricist',
    '曲作者' : 'Composer',
    '专辑UPC' : 'UPC',
    '歌曲ISRC' : 'ISRC',
    '版权公司专辑编码' : 'Licensor Album-Code',
    '版权公司歌曲编码' : 'Licensor Code',
    '版权公司名' : 'Licensor',
    '原始版权公司' : 'Label',
    '词授权有效期开始' : 'Lyrics Right Start Date',
    '词授权有效期结束' : 'Lyrics Right End Date',
    '曲授权有效期开始' : 'Composing Right Start Date',
    '曲授权有效期结束' : 'Composing Right End Date',
    '邻接权授权有效期开始' : 'Recording Right Start Date',
    '邻接权授权有效期结束' : 'Recording Right End Date',
    '词份额' : 'Right Share of Writer',
    '曲份额' : 'Right Share of Composer',
    '邻接权份额' : 'Right Share of Recorder',
    '歌曲付费状态' : 'Charge Type',
    '单价' : 'Retail Price',
    'IOS销量' : 'Sales_IOS',
    '非IOS销量' : 'Sales_Others',
    '单曲订购金额' : "Single Track's Perchase - Revenue",
    '单曲订购收入分成' : 'License Fees - Single Track Purchase',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)'
    })
    return df

def rename_song_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
 #   '结算期间' : 'Period',
 #   '平台' : 'Platform',
 #   '中央曲库歌曲ID' : 'TMEID',
 #   '歌曲名' : 'Song',
 #   '歌曲ISRC' : 'ISRC',
 #   '歌手名' : 'Artist',
 #   '词作者' : 'Lyricist',
 #   '曲作者' : 'Composer',
 #   '专辑名' : 'Album',
 #   '专辑UPC' : 'UPC',
 #   '版权公司名' : 'Licensor',
 #   '原始版权公司' : 'Label',
 #   '词授权有效期开始' : 'Lyrics Right Start Date',
 #   '词授权有效期结束' : 'Lyrics Right End Date',
 #   '曲授权有效期开始' : 'Composing Right Start Date',
 #   '曲授权有效期结束' : 'Composing Right End Date',
 #   '邻接权授权有效期开始' : 'Recording Right Start Date',
 #   '邻接权授权有效期结束' : 'Recording Right End Date',
 #   '版权公司专辑编码' : 'Licensor Album-Code',
 #   '版权公司歌曲编码' : 'Licensor Code',
 #   '词份额' : 'Right Share of Writer',
 #   '曲份额' : 'Right Share of Composer',
 #   '邻接权份额' : 'Right Share of Recorder',
 #   '歌曲付费状态' : 'Charge Type',
 #   '广告收入分成-免模-使用量' : '(unclear)',
 #   '广告收入分成-非免模-使用量' : 'License Fees - Free Music Service-Free Mode-Number of Content Used',
  #  '歌曲驱动金额' : 'License Fees - Free Music Service-Non-free Mode-Number of Content Used',
  #  '基本包月收入分成-使用量' : 'Subscription Music Service(Basic)',
  #  '高级包月收入分成-使用量' : 'Subscription Music Service(Senior)',
  #  '打榜收入' : 'MuCoin & Gift',
  #  '广告收入分成-免模' : 'License Fees for the Ad-Supported Service—Free Mode',
  #  '广告收入分成-非免模' : 'License Fees for the Ad-Supported Service—Non-free Mode',
  #  '基本包月收入分成' : 'License Fees - Subscription Music Service(Basic)',
  #  '高级包月收入分成' : 'License Fees - Subscription Music Service(Senior)',
  #  '打榜收入分成' : 'License Fees - MuCoin & Gift',
  #  'CP分成收入' : 'License Fees - Total' 
    '结算期间' : 'Period',
    '平台' : 'Platform',
    '中央曲库歌曲ID' : 'TMEID',
    '歌曲名' : 'Song',
    '歌曲ISRC' : 'ISRC',
    '歌手名' : 'Artist',
    '词作者' : 'Lyricist',
    '曲作者' : 'Composer',
    '专辑名' : 'Album',
    '专辑UPC' : 'UPC',
    '版权公司名' : 'Licensor',
    '原始版权公司' : 'Label',
    '词授权有效期开始' : 'Lyrics Right Start Date',
    '词授权有效期结束' : 'Lyrics Right End Date',
    '曲授权有效期开始' : 'Composing Right Start Date',
    '曲授权有效期结束' : 'Composing Right End Date',
    '邻接权授权有效期开始' : 'Recording Right Start Date',
    '邻接权授权有效期结束' : 'Recording Right End Date',
    '版权公司专辑编码' : 'Licensor Album-Code',
    '版权公司歌曲编码' : 'Licensor Code',
    '词份额' : 'Right Share of Writer',
    '曲份额' : 'Right Share of Composer',
    '邻接权份额' : 'Right Share of Recorder',
    '歌曲付费状态' : 'Charge Type',
    '广告收入分成-免模-使用量' : 'License Fees - Free Music Service-Free Mode-Number of Content Used',
    '广告收入分成-非免模-使用量' : 'License Fees - Free Music Service-Non-free Mode-Number of Content Used',
    '基本包月收入分成-使用量' : 'Subscription Music Service(Basic)',
    '高级包月收入分成-使用量' : 'Subscription Music Service(Senior)',
    '打榜收入' : 'MuCoin & Gift',
    '广告收入分成-免模' : 'License Fees for the Ad-Supported Service—Free Mode',
    '广告收入分成-非免模' : 'License Fees for the Ad-Supported Service—Non-free Mode',
    '基本包月收入分成' : 'License Fees - Subscription Music Service(Basic)',
    '高级包月收入分成' : 'License Fees - Subscription Music Service(Senior)',
    '打榜收入分成' : 'License Fees - MuCoin & Gift',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)'
    })
    return df

def rename_aiting_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
    '结算期间' : 'Period',
    '平台' : 'Platform',
    '渠道商' : 'Distributor',
    '中央曲库ID' : 'TMEID',
    '歌曲名' : 'Song',
    '歌曲ISRC' : 'ISRC',
    '歌手名' : 'Artist',
    '词作者' : 'Lyricist',
    '曲作者' : 'Composer',
    '专辑名' : 'Album',
    '专辑UPC' : 'UPC',
    '版权公司名' : 'Licensor',
    '原始版权公司' : 'Label',
    '词授权有效期开始' : 'Lyrics Right Start Date',
    '词授权有效期结束' : 'Lyrics Right End Date',
    '曲授权有效期开始' : 'Composing Right Start Date',
    '曲授权有效期结束' : 'Composing Right End Date',
    '邻接权授权有效期开始' : 'Recording Right Start Date',
    '邻接权授权有效期结束' : 'Recording Right End Date',
    '版权公司专辑编码' : 'Licensor Album-Code',
    '版权公司歌曲编码' : 'Licensor Code',
    '词份额' : 'Right Share of Writer',
    '曲份额' : 'Right Share of Composer',
    '邻接权份额' : 'Right Share of Recorder',
    '歌曲付费状态' : 'Charge Type',
    '广告收入分成-使用量' : 'Free Music Service',
    '包月收入分成-使用量' : 'Consumption - Subscription Music Service',
    '广告收入分成' : 'License Fees - Free Music Service',
    '包月收入分成' : 'Licensee Fees - Subscription Music Service',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)'
    })
    return df

def rename_mv_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
    '结算期间' : 'Period',
    '平台' : 'Platform',
    '中央曲库MVID' : 'MVID',
    'MV名' : 'MV',
    '歌手名' : 'Artist',
    '歌曲ISRC' : 'ISRC',
    '专辑UPC' : 'UPC',
    '版权公司名' : 'Licensor',
    '原始版权公司' : 'Label',
    '授权有效期开始' : 'Start Time of Authorization',
    '授权有效期结束' : 'End Time of Authorization',
    '邻接权比例' : 'Share of Recording Right',
    '广告收入分成-非免模-MV使用量' : 'License Fees - Free Music Service-Non-free Mode-Consumption of MV',
    '广告收入分成-非免模' : 'License Fees for the Ad-Supported Service—Non-free Mode',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)'
    })
    return df

def rename_digalbum_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
    '结算期间' : 'Period',
    '平台' : 'Platform',
    '日期' : 'Date',
    '中央曲库专辑ID' : 'Album_ID',
    '专辑名' : 'Album',
    '歌手名' : 'Artist',
    '专辑UPC' : 'UPC',
    '版权公司专辑编码' : 'Licensor Album-Code',
    '词份额' : 'Right Share of Writer',
    '曲份额' : 'Right Share of Composer',
    '邻接权份额' : 'Right Share of Recorder',
    '单价' : 'Retail Price',
    'IOS销量' : 'Sales_IOS',
    '非IOS销量' : 'Sales_Others',
    '数字专辑销售金额' : 'Total Sales Amount of Digital Album',
    '数字专辑销售收入分成' : 'Chargeable Digital Album Service',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)',
    '中央曲库歌曲ID' : 'TMEID',
    '曲作者' : 'Composer',
    '曲授权有效期开始' : 'Composing Right Start Date',
    '曲授权有效期结束' : 'Composing Right End Date',
    '歌曲ISRC' : 'ISRC',
    '歌曲名' : 'Song',
    '词作者' : 'Lyricist',
    '词授权有效期开始' : 'Lyrics Right Start Date',
    '词授权有效期结束' : 'Lyrics Right End Date'
    })
    return df

def rename_k_columns(df_input:pd.DataFrame):
    df = df_input.copy()
    df = df.rename(columns={
    '结算期间' : 'Period',
    '平台' : 'Platform',
    '中央曲库歌曲ID' : 'TMEID',
    '歌曲名' : 'Song',
    '歌曲ISRC' : 'ISRC',
    '歌手名' : 'Artist',
    '词作者' : 'Lyricist',
    '曲作者' : 'Composer',
    '专辑名' : 'Album',
    '专辑UPC' : 'UPC',
    '版权公司名' : 'Licensor',
    '原始版权公司' : 'Label',
    '词授权有效期开始' : 'Lyrics Right Start Date',
    '词授权有效期结束' : 'Lyrics Right End Date',
    '曲授权有效期开始' : 'Composing Right Start Date',
    '曲授权有效期结束' : 'Composing Right End Date',
    '邻接权授权有效期开始' : 'Recording Right Start Date',
    '邻接权授权有效期结束' : 'Recording Right End Date',
    '版权公司专辑编码' : 'Licensor Album-Code',
    '版权公司歌曲编码' : 'Licensor Code',
    '词份额' : 'Right Share of Writer',
    '曲份额' : 'Right Share of Composer',
    '邻接权份额' : 'Right Share of Recorder',
    '按次分成-词使用量' : 'License Fees-Per Play-Consumption of Lyrics',
    '按次分成-曲使用量' : 'License Fees-Per Play-Consumption of Composition',
    '按次分成-邻接权使用量-原版音源' : 'License Fees-Per Play-Consumption of Recording-Original Version',
    '按次分成-邻接权使用量-版权方提供伴奏' : 'License Fees-Per Play-Consumption of Recording-Karaoke Version Provided by Licensor',
    '按次分成-邻接权使用量-依据版权方提供音源制作伴奏' : 'License Fees-Per Play-Consumption of Recording-Karaoke Version Processed by TME',
    '按次分成' : 'License Fees-Per Play',
    'CP分成收入' : 'License Fees - Total',
    '币种' : 'Currency',
    '汇率' : 'Exchange Rate',
    'CP分成收入(CNY)' : 'License Fee(CNY)'
    })
    return df


def fill_missing_exchange_rates(main_df, Lookup_fx):
    """
    Fills missing 'exchange rate' and 'currency' values in main_df using data from Lookup_fx.

    Parameters:
        main_df (pd.DataFrame): The main DataFrame with columns 'platform', 'period', 'currency', 
                                'exchange rate', 'Fee (local currency)', and 'Fee (CNY)'.
        Lookup_fx (pd.DataFrame): The lookup DataFrame with columns 'platform', 'period', 
                                  'currency', and 'exchange rate'.

    Returns:
        pd.DataFrame: The updated main_df with missing 'exchange rate' and 'currency' filled in.
    """
    # Merge the two datasets on 'platform', 'period', and 'currency'
    merged_df = pd.merge(
        main_df,
        Lookup_fx,
        on=['Platform', 'Period'],
        how='left',  # Use 'left' to keep all rows from main_df
        suffixes=('', '_lookup')  # Add suffix to columns from Lookup_fx
    )

    #print(main_df.columns)
    #print(Lookup_fx.columns)
    #print(merged_df.columns)

    # Fill missing 'exchange rate' and 'currency' in main_df with values from Lookup_fx. 
    merged_df['Exchange Rate'] = pd.to_numeric(
        merged_df['Exchange Rate'].fillna(merged_df['Exchange Rate_lookup']),
        errors='coerce'
    )
    merged_df['Currency'] = merged_df['Currency'].fillna(merged_df['Currency_lookup'])

    # Drop the extra columns added during the merge
    merged_df.drop(columns=['Exchange Rate_lookup', 'Currency_lookup','License Fee(CNY)'], inplace=True)

    #if 'Fee (CNY)' in merged_df.columns:
    merged_df.rename(columns={'Fee': 'Fee (local currency)'}, inplace=True)
    merged_df['Fee'] = merged_df['Fee (local currency)'] * merged_df['Exchange Rate']
    merged_df['Fee'] = merged_df['Fee'].fillna(merged_df['Fee (local currency)'])
   

    # Return the updated DataFrame
    return merged_df


JOOX_PLATFORM_BY_CURRENCY = {
    'HKD': 'JOOX-HK',
    'IDR': 'JOOX-ID',
    'MYR': 'JOOX-MY',
    'THB': 'JOOX-TH',
}
CNY_PLATFORMS = ['QQMusic', 'Kugou', 'Kuwo']  # user: QQ, Kugou, Kugo


def _first_nonempty(series):
    for value in series:
        if pd.notna(value) and str(value).strip() not in ('', 'nan', 'None'):
            return value
    return None


def _period_text(value, filename):
    if value is not None:
        match = re.search(r'\d{8}-\d{8}', str(value))
        if match:
            return match.group()
    return find_date_range(filename)


def _pick_column(columns, primary, fallback):
    if primary in columns:
        return primary
    if fallback in columns:
        return fallback
    return None


def collect_joox_fx_from_folders(folders):
    primary = ('结算期间', '币种', '汇率') if chinese == '1' else ('Period', 'Currency', 'Exchange Rate')
    fallback = ('Period', 'Currency', 'Exchange Rate') if chinese == '1' else ('结算期间', '币种', '汇率')
    collected = []
    for folder in sorted(folders):
        if not os.path.isdir(folder):
            continue
        for name in sorted(os.listdir(folder)):
            if 'joox' not in name.lower() or name.startswith('.'):
                continue
            if not name.lower().endswith(('.xlsx', '.xls')):
                continue
            path = os.path.join(folder, name)
            try:
                header_df = pd.read_excel(path, sheet_name=0, nrows=0)
            except Exception as exc:
                print(f'  WARNING: could not read {name}: {exc}')
                continue
            cols = list(header_df.columns)
            period_col = _pick_column(cols, primary[0], fallback[0])
            currency_col = _pick_column(cols, primary[1], fallback[1])
            rate_col = _pick_column(cols, primary[2], fallback[2])
            usecols = [c for c in (period_col, currency_col, rate_col) if c]
            if not usecols:
                print(f'  WARNING: no Period/Currency/Rate columns in {name}')
                continue
            df = pd.read_excel(path, sheet_name=0, usecols=usecols)
            period = _period_text(None if period_col is None else _first_nonempty(df[period_col]), name)
            currency = None if currency_col is None else _first_nonempty(df[currency_col])
            rate = None if rate_col is None else _first_nonempty(df[rate_col])
            if currency is not None:
                currency = str(currency).strip().upper()
            try:
                rate = float(rate) if rate is not None else None
            except (TypeError, ValueError):
                rate = None
            rec = {
                'file': name,
                'period': period,
                'currency': currency,
                'rate': rate,
            }
            collected.append(rec)
            print(
                f'  {name}\n'
                f'    Period {period}    Currency {currency}    Exchange Rate {fmt_money(rate) if rate is not None else None}'
            )
    return collected


def _fmt_fx_rate(x):
    x = float(x)
    if abs(x - round(x)) < 1e-12:
        return str(int(round(x)))
    return f'{x:.10f}'.rstrip('0').rstrip('.')


FX_ROW_RE = re.compile(
    r'(QQMusic|Kugou|Kuwo|JOOX-HK|JOOX-ID|JOOX-MY|JOOX-TH),'
    r'(\d{8}-\d{8}),'
    r'([A-Z]{3}),'
    r'([0-9.]+)'
)


def read_fx_lookup_records(path):
    """Read lookup_fx.csv, repairing a glued last line if a previous append missed a newline."""
    with open(path, 'r', encoding='utf-8-sig', newline='') as f:
        raw = f.read()
    records = []
    repaired = 0
    for line in raw.splitlines():
        line = line.strip()
        if not line or line.lower().startswith('platform,'):
            continue
        parts = next(csv.reader([line]))
        if len(parts) == 4:
            records.append(parts)
            continue
        found = [list(m.groups()) for m in FX_ROW_RE.finditer(line.replace(' ', ''))]
        if found:
            repaired += 1
            records.extend(found)
            print(f'  Repaired glued CSV line  : split into {len(found)} rows')
        else:
            print(f'  WARNING: skipped FX line : {line[:80]}')
    return records, repaired


def write_fx_lookup_records(path, records):
    with open(path, 'w', encoding='utf-8', newline='') as f:
        writer = csv.writer(f, lineterminator='\r\n')
        writer.writerow(['Platform', 'Period', 'Currency', 'Exchange Rate'])
        for rec in records:
            writer.writerow(rec)


def update_lookup_fx_from_joox(folders):
    header('Update lookup_fx.csv from Joox files')
    print(f'  Lookup file              : {lookup_fx}')
    if not os.path.isfile(lookup_fx):
        close_log()
        raise FileNotFoundError(f'FX lookup not found: {lookup_fx}')

    records, repaired = read_fx_lookup_records(lookup_fx)
    df_fx = pd.DataFrame(records, columns=['Platform', 'Period', 'Currency', 'Exchange Rate'])
    df_fx['Exchange Rate'] = pd.to_numeric(df_fx['Exchange Rate'], errors='coerce')
    existing_period_currency = set(
        zip(df_fx['Period'].astype(str), df_fx['Currency'].astype(str).str.upper())
    )
    existing_platform_period = set(
        zip(df_fx['Platform'].astype(str), df_fx['Period'].astype(str))
    )
    print(f'  Existing FX rows         : {fmt_int(len(df_fx))}')
    if repaired:
        print(f'  Glued lines repaired     : {fmt_int(repaired)}')

    collected = collect_joox_fx_from_folders(folders)
    print(f'  Joox files read          : {fmt_int(len(collected))}')

    new_rows = []
    added_joox_by_period = {}
    seen_period_currency = set()
    for rec in collected:
        period, currency, rate = rec['period'], rec['currency'], rec['rate']
        if not period or not currency or rate is None:
            print(f'  SKIP incomplete          : {rec["file"]}')
            continue
        key = (str(period), str(currency))
        if key in seen_period_currency:
            continue
        seen_period_currency.add(key)
        platform = JOOX_PLATFORM_BY_CURRENCY.get(currency)
        if platform is None:
            print(f'  SKIP unknown currency    : {currency} in {rec["file"]}')
            continue
        if key in existing_period_currency:
            print(f'  Already present          : {period} / {currency}')
            continue
        new_rows.append({
            'Platform': platform,
            'Period': period,
            'Currency': currency,
            'Exchange Rate': rate,
        })
        existing_period_currency.add(key)
        added_joox_by_period.setdefault(period, []).append(platform)
        print(f'  ADD  {platform:<8}  {period}  {currency:<4}  {fmt_money(rate)}')

    periods_with_four = [p for p, plats in added_joox_by_period.items() if len(set(plats)) >= 4]
    periods_for_cny = list(added_joox_by_period.keys())
    for period in periods_for_cny:
        if period not in periods_with_four:
            print(
                f'  Note                     : {period} got {len(set(added_joox_by_period[period]))} Joox currencies; '
                'still adding QQMusic/Kugou/Kuwo if missing'
            )
        for platform in CNY_PLATFORMS:
            if (platform, str(period)) in existing_platform_period:
                print(f'  Already present          : {platform} / {period} / CNY')
                continue
            new_rows.append({
                'Platform': platform,
                'Period': period,
                'Currency': 'CNY',
                'Exchange Rate': 1,
            })
            existing_platform_period.add((platform, str(period)))
            print(f'  ADD  {platform:<8}  {period}  CNY   1')

    if new_rows or repaired:
        for row in new_rows:
            records.append([
                row['Platform'],
                row['Period'],
                row['Currency'],
                _fmt_fx_rate(row['Exchange Rate']),
            ])
        write_fx_lookup_records(lookup_fx, records)
        print(f'  New rows written         : {fmt_int(len(new_rows))}')
        print(f'  Lookup saved             : {lookup_fx}')
    else:
        print('  New rows written         : 0  (lookup unchanged)')
    return df_fx

update_lookup_fx_from_joox(subdirs)



  TME quarterly processing
  Run started              : 2026-08-19 22:57:41
  Contract                 : 0044
  Quarter                  : 2026 Q2
  Chinese headers          : 1
  Output file              : TME_0044_2026 Q2.xlsx
  Run log                  : TME_0044_2026 Q2_run_log_20260819_225741.txt

  Update lookup_fx.csv from Joox files
  Lookup file              : ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/support/lookup_fx.csv
  Existing FX rows         : 161
  song_outside_detail_CON02-TME00-20260127-0044_jooxhk_327570544_20260401-20260430-0512013914.xlsx
    Period 20260401-20260430    Currency HKD    Exchange Rate 0.883
  song_outside_detail_CON02-TME00-20260127-0044_jooxid_327570544_20260401-20260430-0512011647.xlsx
    Period 20260401-20260430    Currency IDR    Exchange Rate 0.0004
  song_outside_detail_CON02-TME00-20260127-0044_jooxmy_327570544_20260401-20260430-0512023740.xlsx
    Period 20260401-20260430    Currency MYR    Exchange Rate 1.7157
  song_out

,Platform,Period,Currency,Exchange Rate
0,QQMusic,20241001-20241031,CNY,1.0000
1,Kugou,20241001-20241031,CNY,1.0000
2,Kuwo,20241001-20241031,CNY,1.0000
3,JOOX-HK,20241001-20241031,HKD,0.9018
4,JOOX-ID,20241001-20241031,IDR,0.0005
...,...,...,...,...
156,Kugou,20260501-20260531,CNY,1.0000
157,Kuwo,20260501-20260531,CNY,1.0000
158,QQMusic,20260601-20260630,CNY,1.0000
159,Kugou,20260601-20260630,CNY,1.0000


In [52]:
ensure_logging()
df_single = read_and_combine_files(base_dir = subdirs, keyword_file="single")
if chinese == "1":
        print(df_single.columns)
        df_single = rename_single_columns(df_single)
        print(df_single.columns)
df_single = add_columns_and_modify_columns_to_single_df(df_input = df_single)
df_single.head()

print(f"The combined DataFrame has {df_single.shape[0]} rows and {len(df_single.columns)} columns.")


Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/single_outside_detail_CON02-TME00-20260127-0044_qq_327570544_20260401-20260430-0512024846.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/single_outside_detail_CON02-TME00-20260127-0044_kg_327570544_20260401-20260430-0512024651.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/single_outside_detail_CON02-TME00-20260127-0044_kw_327570544_20260401-20260430-0512025025.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 05
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 05/single_outside_detail_CON02-TME

In [53]:
ensure_logging()
df_song = read_and_combine_files(base_dir = subdirs, keyword_file="song")
if chinese == "1":
        df_song = rename_song_columns(df_song)
df_song = add_columns_and_modify_columns_to_song_df(df_input = df_song)
df_song.head()

print(f"The combined DataFrame has {df_song.shape[0]} rows and {len(df_song.columns)} columns.")


Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/song_outside_detail_CON02-TME00-20260127-0044_kw_327570544_20260401-20260430-0512015814.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/song_outside_detail_CON02-TME00-20260127-0044_jooxth_327570544_20260401-20260430-0512011641.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/song_outside_detail_CON02-TME00-20260127-0044_jooxid_327570544_20260401-20260430-0512011647.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/song_outside_detail_CON02-TME00-20260127-0044_jooxmy_327570544_20260401-20260430-0512023740.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME

In [54]:
ensure_logging()
# Files "song" - part B 

def create_subset(df, common_columns, specific_columns):
    """
    Create a subset of the DataFrame using common and specific columns.
    
    :param df: The original DataFrame.
    :param common_columns: List of columns common to all subsets.
    :param specific_columns: List of columns specific to each subset.
    :return: A subset DataFrame with selected columns.
    """
    # Combine common columns with specific columns
    columns_to_select = common_columns + specific_columns
    # Return the subset DataFrame
    return df[columns_to_select]

# Function to process each subset DataFrame
def process_dataframe(df, rename_dict, product, product_detail=None):
    df = df.rename(columns=rename_dict)
    df['Product'] = product
    if product_detail:
        df['Product (Detail)'] = product_detail
    return df

# List of common columns
common_columns = [
    'Period',
    'Platform',
    'TMEID',
    'Song',
    'ISRC',
    'Artist',
    'Lyricist',
    'Composer',
    'Album',
    'UPC',
    'Licensor',
    'Label',
    'Lyrics Right Start Date',
    'Lyrics Right End Date',
    'Composing Right Start Date',
    'Composing Right End Date',
    'Recording Right Start Date',
    'Recording Right End Date',
    'Licensor Album-Code',
    'Licensor Code',
    'Right Share of Writer',
    'Right Share of Composer',
    'Right Share of Recorder',
    'Charge Type'
]

# Free Music Service - Free Mode
if year == 2024 or year == 2025 or year == 2026:
    subset_df_4_6_free1 = create_subset(
        df_song,
        common_columns,
        [
            'License Fees - Free Music Service-Free Mode-Number of Content Used',
            'License Fees for the Ad-Supported Service—Free Mode'
        ]
    )
else:
    subset_df_4_6_free1 = create_subset(
        df_song,
        common_columns,
        [
            'Free Music Service',
            'License Fees - Free Music Service'
        ]
    ) 


# Free Music Service - Non-free Mode
if year == 2024 or year == 2025 or year == 2026:
    subset_df_4_6_free2 = create_subset(
        df_song,
        common_columns,
        [
            'License Fees - Free Music Service-Non-free Mode-Number of Content Used',
            'License Fees for the Ad-Supported Service—Non-free Mode'
        ]
    )

# Subscription Music Service (Basic)
subset_df_4_6_subscription = create_subset(
    df_song,
    common_columns,
    [
        'Subscription Music Service(Basic)',
        'License Fees - Subscription Music Service(Basic)'
    ]
)

# Subscription Music Service (Senior)
subset_df_4_6_subscription_premium = create_subset(
    df_song,
    common_columns,
    [
        'Subscription Music Service(Senior)',
        'License Fees - Subscription Music Service(Senior)'
    ]
)

# MuCoin & Gift
subset_df_4_6_MUcoins = create_subset(
    df_song,
    common_columns,
    [
        'MuCoin & Gift',
        'License Fees - MuCoin & Gift'
    ]
)

# List of dictionaries with DataFrame information
if year == 2024 or year == 2025 or year == 2026: 
    dataframes_info = [
        {
            'df': subset_df_4_6_free1,
            'rename_dict': {'License Fees - Free Music Service-Free Mode-Number of Content Used': 'Units', 'License Fees for the Ad-Supported Service—Free Mode': 'Fee'},
            'product': 'Ad-supported',
            'product_detail': 'Free Music Service-Free Mode'
        },
        {
            'df': subset_df_4_6_free2,
            'rename_dict': {'License Fees - Free Music Service-Non-free Mode-Number of Content Used': 'Units', 'License Fees for the Ad-Supported Service—Non-free Mode': 'Fee'},
            'product': 'Ad-supported',
            'product_detail': 'Free Music Service-Non-free Mode'
        },
        {
            'df': subset_df_4_6_subscription,
            'rename_dict': {'Subscription Music Service(Basic)': 'Units', 'License Fees - Subscription Music Service(Basic)': 'Fee'},
            'product': 'Subscription (Basic)'
        },
        {
            'df': subset_df_4_6_subscription_premium,
            'rename_dict': {'Subscription Music Service(Senior)': 'Units', 'License Fees - Subscription Music Service(Senior)': 'Fee'},
            'product': 'Subscription (Premium)'
        },
        {
            'df': subset_df_4_6_MUcoins,
            'rename_dict': {'MuCoin & Gift': 'Units', 'License Fees - MuCoin & Gift': 'Fee'},
            'product': 'MU Coin'
        }
    ]
else: 
    dataframes_info = [
        {
            'df': subset_df_4_6_free1,
            'rename_dict': {'Free Music Service': 'Units', 'License Fees - Free Music Service': 'Fee'},
            'product': 'Ad-supported',
        },
        {
            'df': subset_df_4_6_subscription,
            'rename_dict': {'Subscription Music Service(Basic)': 'Units', 'License Fees - Subscription Music Service(Basic)': 'Fee'},
            'product': 'Subscription (Basic)'
        },
        {
            'df': subset_df_4_6_subscription_premium,
            'rename_dict': {'Subscription Music Service(Senior)': 'Units', 'License Fees - Subscription Music Service(Senior)': 'Fee'},
            'product': 'Subscription (Premium)'
        },
        {
            'df': subset_df_4_6_MUcoins,
            'rename_dict': {'MuCoin & Gift': 'Units', 'License Fees - MuCoin & Gift': 'Fee'},
            'product': 'MU Coin'
        }
    ]

# Process each DataFrame and store the result in a list
processed_dfs = [process_dataframe(info['df'], info['rename_dict'], info['product'], info.get('product_detail')) for info in dataframes_info]

# Convert 'Fee' to float specifically for MU Coin DataFrame
processed_dfs[-1]['Fee'] = processed_dfs[-1]['Fee'].astype(float)

# Concatenate all processed DataFrames
df_song_modified = pd.concat(processed_dfs, ignore_index=True)

print(f"The combined DataFrame has {df_song_modified.shape[0]} rows and {len(df_song_modified.columns)} columns.")


The combined DataFrame has 14700 rows and 28 columns.


In [55]:
ensure_logging()
df_aiting = read_and_combine_files(base_dir = subdirs, keyword_file="aiting")
if not df_aiting.empty:  
    if chinese == "1":
        df_aiting = rename_aiting_columns(df_aiting)
    df_aiting.head()
    print(f"The combined DataFrame has {df_aiting.shape[0]} rows and {len(df_aiting.columns)} columns.")


Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/aiting_song_outside_detail_CON02-TME00-20260127-0044_327570544_20260401-20260430-0512013004.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 05
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 05/aiting_song_outside_detail_CON02-TME00-20260127-0044_329740052_20260501-20260531-0624015550.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 06
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 06/aiting_song_outside_detail_CON02-TME00-20260127-0044_331091837_20260601-20260630-0712030334.xlsx
The com

In [56]:
ensure_logging()
# Part B for "aiting" file

# Dictionary to define renaming and product details
columns_info = {
    'Free Music Service': {
        'rename': {'Free Music Service': 'Units', 'License Fees - Free Music Service': 'Fee'},
        'product': 'Ad-supported'
    },
    'Consumption - Subscription Music Service': {
        'rename': {'Consumption - Subscription Music Service': 'Units', 'Licensee Fees - Subscription Music Service': 'Fee'},
        'product': 'Subscription'
    }
}

# Function to process and rename columns, add 'Product' column
def process_and_label(df, column_key):
    df = df.rename(columns=columns_info[column_key]['rename'])
    df['Product'] = columns_info[column_key]['product']
    return df

# Process both DataFrames
subset_df_7_free = process_and_label(df_aiting[[
    'Period', 'Platform', 'Distributor', 'TMEID', 'Song', 'ISRC', 'Artist', 'Lyricist', 'Composer', 
    'Album', 'UPC', 'Licensor', 'Label', 'Lyrics Right Start Date', 'Lyrics Right End Date', 
    'Composing Right Start Date', 'Composing Right End Date', 'Recording Right Start Date', 
    'Recording Right End Date', 'Licensor Album-Code', 'Licensor Code', 'Right Share of Writer', 
    'Right Share of Composer', 'Right Share of Recorder', 'Charge Type', 'Free Music Service', 
    'License Fees - Free Music Service'
]], 'Free Music Service')

subset_df_7_subscription = process_and_label(df_aiting[[
    'Period', 'Platform', 'Distributor', 'TMEID', 'Song', 'ISRC', 'Artist', 'Lyricist', 'Composer', 
    'Album', 'UPC', 'Licensor', 'Label', 'Lyrics Right Start Date', 'Lyrics Right End Date', 
    'Composing Right Start Date', 'Composing Right End Date', 'Recording Right Start Date', 
    'Recording Right End Date', 'Licensor Album-Code', 'Licensor Code', 'Right Share of Writer', 
    'Right Share of Composer', 'Right Share of Recorder', 'Charge Type', 
    'Consumption - Subscription Music Service', 'Licensee Fees - Subscription Music Service'
]], 'Consumption - Subscription Music Service')

# Concatenate the processed DataFrames
df_aiting_modified = pd.concat([subset_df_7_free, subset_df_7_subscription], ignore_index=True)
#print(df_aiting_modified)


In [57]:
ensure_logging()
df_mv = read_and_combine_files(base_dir = subdirs, keyword_file="mv")
if not df_mv.empty:
    if chinese == "1":
        # print(df_mv)
        df_mv = rename_mv_columns(df_mv)
        # print(df_mv)
    df_mv = add_columns_and_modify_columns_to_mv_df(df_input = df_mv)
    df_mv_album = pd.read_csv(lookup_mv_album)
    df_mv = merge(df_mv, df_mv_album, 'Song')
    df_mv.head()
    print(f"The combined DataFrame has {df_mv.shape[0]} rows and {len(df_mv.columns)} columns.")


Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/mv_outside_detail_ver_CON02-TME00-20260127-0044_qq_327570544_20260401-20260430-0512025459.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/mv_outside_detail_ver_CON02-TME00-20260127-0044_kw_327570544_20260401-20260430-0512025424.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/mv_outside_detail_ver_CON02-TME00-20260127-0044_kg_327570544_20260401-20260430-0512025354.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 05
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 05/mv_outside_detail_ver_CON02-TME

In [58]:
ensure_logging()
df_digalbum = read_and_combine_files(base_dir = subdirs, keyword_file="digital")
if not df_digalbum.empty:
    if chinese == "1":
        df_digalbum = rename_digalbum_columns(df_digalbum)
    df_digalbum = add_columns_and_modify_columns_to_digalbum_df(df_input = df_digalbum)
    df_digalbum.head()
    print(f"The combined DataFrame has {df_digalbum.shape[0]} rows and {len(df_digalbum.columns)} columns.")


Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/digital_album_outside_detail_CON02-TME00-20260127-0044_kg_327570544_20260401-20260430-0512024543.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/digital_album_outside_detail_CON02-TME00-20260127-0044_qq_327570544_20260401-20260430-0512024530.xlsx
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/digital_album_outside_detail_CON02-TME00-20260127-0044_kw_327570544_20260401-20260430-0512024555.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 05
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 05/digital_al

In [59]:
ensure_logging()
df_k = read_and_combine_files(base_dir = subdirs, keyword_file="k_outside")
if not df_k.empty:  
    if chinese == "1":
        df_k = rename_k_columns(df_k)
    df_k = add_columns_and_modify_columns_to_k_df(df_input = df_k)
    df_k.head()
    print(f"The combined DataFrame has {df_k.shape[0]} rows and {len(df_k.columns)} columns.")


Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 04/k_outside_detail_CON02-TME00-20260127-0044_327570544_20260401-20260430-0512051654.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 05
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 05/k_outside_detail_CON02-TME00-20260127-0044_329740052_20260501-20260531-0624024914.xlsx
Accessing folder: ../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 06
../../50 KM Group/Royalties/Statements/Karen/Tencent - TME/Contract_0044/2026 Q2/as supplied by TME/2026 06/k_outside_detail_CON02-TME00-20260127-0044_331091837_20260601-20260630-0712071402.xlsx
The combined DataFrame has 3657 rows 

In [60]:
ensure_logging()
# Combine all product types into one DataFrame. pd.concat aligns the columns
# itself, so no column is pre-filled with NA. Frames without rows are dropped:
# they carry no dtype information and would only blur the dtypes of the result.
def combine(frames):
    frames = [df for df in frames if not df.empty]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

df_final1 = combine([
    df_k,
    df_digalbum,
    df_mv,
    df_single,
    df_song_modified,
    df_aiting_modified,
])


# Assuming main_df and Lookup_fx are already defined
df_fx = pd.read_csv(lookup_fx)
df_final = fill_missing_exchange_rates(df_final1, df_fx)

# Add 'Contract' and 'Quarter' columns
df_final['Contract'] = contract
df_final['Quarter'] = quarter
df_platform = pd.read_csv(lookup_platform)
df_final = pd.merge(df_final, df_platform, on='Platform', how='left')

df_final = df_final.rename(columns={
    'ISRC': 'ISRC (orig)',
    'Album': 'Album (orig)',
    'Song': 'Song (orig)',
    'TME MVID': 'MVID',
    'Product (Detail)': 'Product (detail)',
})

template_tme = (
    '../../50 KM Group/Royalties/Statements/Karen/'
    'Tencent - TME/Combined Statements/TME_royalties_2022Q4_2026Q2.xlsx'
)
template_cols = list(pd.read_excel(template_tme, sheet_name='data', nrows=0).columns)
missing_cols = [c for c in template_cols if c not in df_final.columns]
extra_cols = [c for c in df_final.columns if c not in template_cols]
for col in missing_cols:
    df_final[col] = pd.NA
df_final = df_final.reindex(columns=template_cols)

print(f"The final DataFrame has {df_final.shape[0]} rows and {len(df_final.columns)} columns.")
print(f"Total fee: {df_final['Fee'].sum()}. Total units: {df_final['Units'].sum()}")

# Save the final DataFrame to Excel
df_final.to_excel(outputfilename, engine='openpyxl', index=False)

header('Save combined statement')
print(f'  Combined workbook        : {outputfilename}')
print(f'  Column template          : {os.path.basename(template_tme)}')
print(f'  {df_final.shape[0]:,} rows × {len(df_final.columns)} columns')
if missing_cols:
    print(f'  Blank columns added      : {len(missing_cols)}  ({", ".join(missing_cols)})')
else:
    print('  Blank columns added      : 0')
if extra_cols:
    print(f'  Extra columns dropped    : {len(extra_cols)}  ({", ".join(str(c) for c in extra_cols)})')
print(f'  Fee                      : {df_final["Fee"].sum():,.4f}')
print(f'  Units                    : {df_final["Units"].sum():,.2f}')

header('Final numbers check')
from openpyxl import load_workbook

bill_label = '本次收入分成合计' if str(chinese) == '1' else 'License Fees (Total)'
bill_files = []
for month_dir in sorted(subdirs):
    if not os.path.isdir(month_dir):
        continue
    for name in sorted(os.listdir(month_dir)):
        if name.startswith('~$'):
            continue
        if 'bill_outside' in name.lower() and name.lower().endswith(('.xlsx', '.xls')):
            bill_files.append(os.path.join(month_dir, name))


def bill_outside_amount(path, label):
    """Read the first sheet and return the amount next to `label`."""
    wb = load_workbook(path, read_only=True, data_only=True)
    try:
        ws = wb[wb.sheetnames[0]]
        header_cols = []
        row_amounts = []
        rows = list(ws.iter_rows(max_col=15, values_only=True))
        for row_idx, row in enumerate(rows):
            for col_idx, val in enumerate(row):
                if val is None or str(val).strip() != label:
                    continue
                nums = [
                    c for c in row[col_idx + 1:]
                    if isinstance(c, (int, float)) and not isinstance(c, bool)
                ]
                if nums:
                    row_amounts.append(float(nums[0]))
                else:
                    header_cols.append((row_idx, col_idx))
        if row_amounts:
            return sum(row_amounts)
        total = 0.0
        found_numeric = False
        for header_row, col_idx in header_cols:
            for row in rows[header_row + 1:]:
                if col_idx >= len(row):
                    continue
                val = row[col_idx]
                if isinstance(val, (int, float)) and not isinstance(val, bool):
                    total += float(val)
                    found_numeric = True
        return total if found_numeric else None
    finally:
        wb.close()


print(f'  Source label             : {bill_label}')
print(f'  bill_outside files       : {len(bill_files)}')

bill_sum = 0.0
missing_label = []
for path in bill_files:
    amount = bill_outside_amount(path, bill_label)
    if amount is None:
        missing_label.append(os.path.basename(path))
        print(f'  {os.path.basename(path):<60}  NOT FOUND')
    else:
        bill_sum += amount
        print(f'  {os.path.basename(path):<60}  {amount:,.4f}')

fee_sum = float(pd.to_numeric(df_final['Fee'], errors='coerce').sum())
delta = fee_sum - bill_sum
print(f'  Sum {bill_label:<18} : {bill_sum:,.4f}')
print(f'  Sum Fee                 : {fee_sum:,.4f}')
print(f'  Delta (Fee − bills)     : {delta:,.4f}')
if not bill_files:
    print('  ALERT: no files with "bill_outside" in the name were found in the monthly source folders.')
elif missing_label:
    print(f'  ALERT: label "{bill_label}" was missing in {len(missing_label)} file(s).')
elif abs(delta) > 0.01:
    print(f'  ALERT: Fee vs bill_outside deviation is {delta:,.4f} CNY (threshold 0.01).')
else:
    print('  Check                    : OK  (|delta| ≤ 0.01 CNY)')

print(f'  Run finished             : {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
close_log()


The final DataFrame has 24469 rows and 49 columns.
Total fee: 840177.1011154389. Total units: 124648933.0

  Save combined statement
  Combined workbook        : ../../50 KM Group/Royalties/Statements/Karen/_output/TME_0044_2026 Q2.xlsx
  Column template          : TME_royalties_2022Q4_2026Q2.xlsx
  24,469 rows × 49 columns
  Blank columns added      : 5  (Entry No, Album&Song, ISRC, Album, Song)
  Fee                      : 840,177.1011
  Units                    : 124,648,933.00

  Final numbers check
  Source label             : 本次收入分成合计
  bill_outside files       : 3
  bill_outside_A2026040608325_CON02-TME00-20260127-0044_202604.xlsx  281,546.1372
  bill_outside_A2026051285444_CON02-TME00-20260127-0044_202605.xlsx  274,543.8743
  bill_outside_A2026060883557_CON02-TME00-20260127-0044_202606.xlsx  284,087.0896
  Sum 本次收入分成合计           : 840,177.1011
  Sum Fee                 : 840,177.1011
  Delta (Fee − bills)     : 0.0000
  Check                    : OK  (|delta| ≤ 0.01 CNY)
  Run 